# Quiet checkers tablebase inspection

Run the quiet-position generator, watch progress, and inspect the packed-state transition maps. The backend stores each round as metadata-keyed maps from source state_key to sets of target state_keys.

In [ ]:
%matplotlib inline

from pathlib import Path
import sys
from importlib import reload

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parents[1]

dev_notebook_dir = repo_root / "notebooks" / "dev"
if str(dev_notebook_dir) not in sys.path:
    sys.path.insert(0, str(dev_notebook_dir))

import quiet_notebook_live
quiet_notebook_live = reload(quiet_notebook_live)

from pycheckers import (
    FORCED_CAPTURE,
    FORCED_CAPTURE_PROMOTION,
    PROMOTION,
    QUIET,
    inspect_game_state,
    run_quiet_graph,
)
from quiet_notebook_live import run_live_quiet_exploration


## Generate

In [ ]:
MAX_ROUNDS = None
MAX_SECONDS = None
MAX_TOTAL_STATES = None
RUN_NAME = None

live_result = run_live_quiet_exploration(
    max_rounds=MAX_ROUNDS,
    max_seconds=MAX_SECONDS,
    max_total_states=MAX_TOTAL_STATES,
    run_name=RUN_NAME,
    pause_sec=0.05,
)

live_result.summary()


## Round Metrics

In [ ]:
rounds_df = live_result.rounds_df()
rounds_df.style


In [ ]:
performance_cols = [
    "round",
    "frontier_in",
    "processed",
    "new_states",
    "duplicate_states",
    "same_round_state_collisions",
    "prior_generation_state_hits",
    "total_states",
    "total_transitions",
    "round_sec",
    "states_per_sec",
]

rounds_df[performance_cols].style


## Tablebase Structure

In [ ]:
summary_df = live_result.tablebase_summary_df()
metadata_maps_df = live_result.metadata_maps_df()

summary_df.T


In [ ]:
metadata_maps_df.style

In [ ]:
metadata_maps_df.pivot(index="round",columns="metadata",values="distinct_target_states").cumsum()

In [ ]:
metadata_maps_df.pivot(index="round",columns="metadata",values="distinct_target_states").cumsum().plot(figsize=(20,4))

## Inspect One State

In [ ]:
state_index = 0
state_key = live_result.state_keys[state_index]
state_info = live_result.inspect_state(state_index)

{
    "state_index": state_info["state_index"],
    "state_key": state_info["state_key"],
    "quiet_successors": None if state_info["quiet_successors"] is None else len(state_info["quiet_successors"]),
    "forced_capture": state_info["forced_capture"],
    "promotion_possible": state_info["promotion_possible"],
    "terminal": state_info["terminal"],
    "metadata_edges": {k: len(v) for k, v in state_info["metadata_successors"].items()},
}


In [ ]:
live_result.show_state(state_index, size=1.5)


## Inspect Raw Maps

In [ ]:
round_number = 1
metadata = QUIET
transition_map = live_result.transition_map(round_number, metadata)
sample_source = next(iter(transition_map), None)

{
    "round": round_number,
    "metadata": metadata,
    "source_states": len(transition_map),
    "edges": sum(len(targets) for targets in transition_map.values()),
    "sample_source": sample_source,
    "sample_targets": [] if sample_source is None else list(transition_map[sample_source])[:10],
}


## DataFrames

In [ ]:
states_df = live_result.states_df()
transitions_df = live_result.transitions_df()
boundary_states_df = live_result.boundary_states_df()
boundary_edges_df = live_result.boundary_edges_df()

In [ ]:
states_df

In [ ]:
transitions_df


In [ ]:
boundary_edges_df
